# Fase 2 — Entendimento dos Dados
**Projeto:** Modelo de Risco de Crédito  
**Autor:** Equipe de Risco e Analytics

In [ ]:
# --- Configurações iniciais ---
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.family"] = "DejaVu Sans"
%matplotlib inline

# Caminhos (relativos ao notebook — que fica em notebooks/clean ou notebooks/executed)
NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))
DADOS_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "credito_tratado.csv")
FIG_PATH = os.path.join(PROJECT_ROOT, "reports", "figures")

print(f"Projeto raiz: {PROJECT_ROOT}")
print(f"Dados:       {DADOS_PATH}")



## 1. Carregamento, dimensão e primeiras linhas

In [ ]:
# ============================================================
# ITEM 1 — Carregar CSV, shape, primeiras linhas
# ============================================================
df = pd.read_csv(DADOS_PATH)

print(f"Número de LINHAS  (clientes):  {df.shape[0]:,}")
print(f"Número de COLUNAS (features): {df.shape[1]}")
print(f"Memória utilizada: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
df.head()



## 2. Tipos de dados, significado e unidade de cada coluna

In [ ]:
# ============================================================
# ITEM 2 — Tipos de dados, significado e unidade de cada coluna
# ============================================================
dicionario = {
    "inadimplente_2anos":         ("Inteiro binário",  "1=inadimplente 90+ DPD em até 2 anos; 0=adimplente",   "Flag (0/1)"),
    "idade":                      ("Inteiro",          "Idade do cliente no momento da avaliação de crédito", "Anos"),
    "renda_mensal":               ("Contínuo",         "Renda mensal informada ou estimada pelo cliente",     "Reais (R$)"),
    "dependentes":                ("Discreto",         "Número de dependentes declarados",                    "Pessoas"),
    "uso_limite_rotativo":        ("Contínuo",         "Proporção do limite rotativo já utilizada",           "Proporção (0–1; pode >1 se estourado)"),
    "razao_divida":               ("Contínuo",         "Relação entre comprometimento financeiro e renda",    "Razão (sem unidade)"),
    "linhas_credito_abertas":     ("Inteiro",          "Qtd. de linhas de crédito ativas no histórico",       "Unidades"),
    "financiamentos_imobiliarios":("Inteiro",          "Número de financiamentos imobiliários registrados",   "Unidades"),
    "atrasos_30_59_dias":         ("Inteiro",          "Qtd. de episódios de atraso 30–59 dias",              "Unidades"),
    "atrasos_60_89_dias":         ("Inteiro",          "Qtd. de episódios de atraso 60–89 dias",              "Unidades"),
    "atrasos_90_mais_dias":       ("Inteiro",          "Qtd. de episódios de atraso >= 90 dias",              "Unidades"),
}

tipos = df.dtypes.reset_index()
tipos.columns = ["Coluna", "Tipo Pandas"]
tipos["Classificação"] = tipos["Coluna"].map(lambda c: dicionario.get(c, ("", "", ""))[0])
tipos["Significado"] = tipos["Coluna"].map(lambda c: dicionario.get(c, ("", "", ""))[1])
tipos["Unidade"] = tipos["Coluna"].map(lambda c: dicionario.get(c, ("", "", ""))[2])

tipos.style.set_properties(**{"text-align": "left"}).hide(axis="index")



## 3. Distribuição do alvo — inadimplente_2anos

In [ ]:
# ============================================================
# ITEM 3 — Distribuição do alvo (inadimplente_2anos)
# ============================================================
alvo = "inadimplente_2anos"

contagem   = df[alvo].value_counts().sort_index()
percentual = (contagem / len(df) * 100).round(2)

tabela_alvo = pd.DataFrame({
    "Classe":        contagem.index.map({0: "Adimplente (0)", 1: "Inadimplente (1)"}),
    "Contagem":      contagem.values,
    "Percentual (%)": percentual.values,
})
display(tabela_alvo.style.hide(axis="index"))

n_adimplentes   = contagem[0]
n_inadimplentes = contagem[1]
razao = n_adimplentes / n_inadimplentes if n_inadimplentes > 0 else np.inf

print(f"Razão de desequilíbrio: 1 INADIMPLENTE para cada {razao:.2f} ADIMPLENTES.")
print(f"Representatividade classe positiva (inadimplentes): {percentual[1]:.2f}%")

# Gráfico
fig, ax = plt.subplots(figsize=(6, 4.5))
colors = ["#2ecc71", "#e74c3c"]
sns.countplot(x=alvo, data=df, ax=ax, palette=colors)
ax.set_title("Distribuição da variável-alvo (inadimplente_2anos)", fontsize=13, pad=15)
ax.set_xlabel("Classe")
ax.set_ylabel("Frequência")
ax.set_xticklabels(["Adimplente (0)", "Inadimplente (1)"])
for p in ax.patches:
    h = p.get_height()
    ax.annotate(f"{h:,}\n({h/len(df)*100:.2f}%)", (p.get_x()+p.get_width()/2, h),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIG_PATH, "fase02_distribuicao_alvo.png"), dpi=120)
plt.show()



## 4. Acurácia do modelo baseline de 1 linha (chuta ninguém calota)

In [ ]:
# ============================================================
# ITEM 4 — Acurácia do modelo de uma linha que chuta "ninguém calota"
# ============================================================
predominante = 0 if contagem[0] >= contagem[1] else 1
acuracia_baseline = (df[alvo] == predominante).mean() * 100

print(f"Regra do modelo baseline: prever SEMPRE {predominante} (Adimplente).")
print(f"Acurácia do modelo de 1 linha (chuta tudo 0): {acuracia_baseline:.2f}%")



## 5. Consequências para a escolha das métricas

In [ ]:
# ============================================================
# ITEM 5 — Consequências para a escolha de métricas
# ============================================================
from IPython.display import Markdown, display

texto = f"""
---

### Consequência imediata do baseline de {acuracia_baseline:.2f}%

Com **{percentual[1]:.2f}%** de positivos contra **{percentual[0]:.2f}%** de negativos, a **ACURÁCIA é inútil como métrica decisória** — um modelo que não faz nada (chuta tudo 0) já entrega {acuracia_baseline:.2f}% de acerto.

> Se usarmos acurácia como objetivo, o algoritmo tenderá a "nunca acusar ninguém", deixando passar Falsos Negativos, que custam **10x mais caro** que os Falsos Positivos.

---

### Métricas que usaremos neste projeto

| Métrica | O que mede | Cuidados que temos que tomar |
|---|---|---|
| **ROC AUC** (≥ 0,85 obrigatório) | Capacidade de **ordenação** do modelo: quão bem ele separa clientes bons de maus pagadores independentemente do ponto de corte. | Pode ser artificialmente alto em dados extremamente desbalanceados; **não usa a matriz de custos FN/FP**, portanto mede performance estatística, não econômica. |
| **Precision (Precisão)** | Dos clientes marcados como "ruins" (negados pelo modelo), quantos **realmente** calotaram? — TP/(TP+FP). | Mede só o lado do "negado" (FP). Um modelo que nega 1 única pessoa com 100% de certeza teria Precision=100% e não serve ao negócio. |
| **Recall / Sensibilidade (Revogação)** | De **todos os verdadeiros calotes** existentes na base, quantos o modelo conseguiu pegar? — TP/(TP+FN). | Se aumentar cegamente (ponto de corte muito baixo), explode o número de FPs e nega muita gente boa, destruindo volume de aprovações. |
| **F1-Score** | Média harmônica entre Precision e Recall. | **Trata FN e FP com mesmo peso**, mas na nossa relação 10:1 isso não reflete a realidade de custo. Usamos só como comparativo, NUNCA como decisório. |
| **Matriz de Confusão + Custo Esperado da Carteira** | `Custo = (N_FN × 10) + (N_FP × 1)` — o custo total da política em R$ (na unidade definida). | **Esta é a métrica de NEGÓCIO.** Depende estritamente do **ponto de corte** escolhido pela Área de Risco. Vamos otimizar sobre ela, não sobre acurácia. |
| **Brier Score / Calibração** | Quão calibradas são as probabilidades: um cliente com score 18% deve ter ~18% de chance real de calote. | Crítico para pricing e para o analista confiar no número que aparece na tela do sistema. |
"""
display(Markdown(texto))



## 6. Valores nulos por coluna

In [ ]:
# ============================================================
# ITEM 6 — Colunas com valores NULOS (qtd e %)
# ============================================================
nulos     = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)

tabela_nulos = pd.DataFrame({
    "Coluna":    nulos.index,
    "Qtd Nulos": nulos.values,
    "% Nulos":   nulos_pct.values,
})
tabela_nulos = tabela_nulos[tabela_nulos["Qtd Nulos"] > 0]     .sort_values("% Nulos", ascending=False)     .reset_index(drop=True)

display(tabela_nulos.style.hide(axis="index"))

# Gráfico
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x="% Nulos", y="Coluna", data=tabela_nulos, ax=ax, palette="Reds_r")
ax.set_title("Percentual de valores nulos por coluna", fontsize=13, pad=15)
for i, v in enumerate(tabela_nulos["% Nulos"].values):
    qtd = tabela_nulos["Qtd Nulos"].iloc[i]
    ax.text(v + 0.1, i, f"{v:.2f}%  (n={qtd:,})", va="center", fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIG_PATH, "fase02_valores_nulos.png"), dpi=120)
plt.show()



## 7. Podemos simplesmente descartar as linhas com nulos?

In [ ]:
# ============================================================
# ITEM 7 — Podemos descartar as linhas com nulos?
# ============================================================
linhas_com_nulo = df.isnull().any(axis=1).sum()
pct_linhas_nulo = (linhas_com_nulo / len(df) * 100).round(2)

tabela_drop = pd.DataFrame([
    ["Total de clientes",                        f"{len(df):,}",      "100,00%"],
    ["Linhas com PELO MENOS 1 nulo",             f"{linhas_com_nulo:,}", f"{pct_linhas_nulo:.2f}%"],
    ["Linhas completas (sem nenhum nulo)",       f"{len(df)-linhas_com_nulo:,}", f"{100-pct_linhas_nulo:.2f}%"],
], columns=["Cenário", "Quantidade", "Percentual"])

display(tabela_drop.style.hide(axis="index"))

msg = f"""
---

### Conclusão sobre drop de linhas

Simplesmente **dropar** as linhas com valor faltante = perder **{linhas_com_nulo:,} clientes ({pct_linhas_nulo:.2f}% da base)**.

Além do volume perdido, o problema de viés é o mais grave:
- O missing em `renda_mensal` (19,82%) **não é aleatório**. Quem NÃO informa a renda em um pedido de crédito já carrega um **sinal de risco** — a ausência da informação é, ela mesma, informação. Dropar essas linhas estaria removendo justamente um perfil de maior risco do treinamento, e o modelo aprenderia enviesado.
- O missing em `dependentes` (2,62%) é menor mas segue o mesmo raciocínio.

### Recomendação (já com base nos números)

| Coluna | Ação sugerida na Fase 3 |
|---|---|
| `renda_mensal` | 1) Criar **flag binária** `renda_ausente` (0/1); 2) **Imputar** o missing com a **mediana** (robusta a outliers). |
| `dependentes`  | 1) Criar **flag binária** `dependentes_ausente` (0/1); 2) **Imputar** com moda (0 ou mediana). |
"""
from IPython.display import Markdown; display(Markdown(msg))



## 8. Outros problemas na base (outliers extremos, duplicatas, etc.)

In [ ]:
# ============================================================
# ITEM 8 — Outros problemas detectados (describe + outliers + outros)
# ============================================================
desc = df.describe().T
desc["missing"]     = df.isnull().sum()
desc["missing_pct"] = (df.isnull().sum() / len(df) * 100).round(2)
print("ESTATÍSTICAS DESCRITIVAS:")
display(desc.round(2))

colunas_num = df.select_dtypes(include=np.number).columns.tolist()
colunas_num.remove(alvo)

p99 = df[colunas_num].quantile(0.99)
p01 = df[colunas_num].quantile(0.01)
mx  = df[colunas_num].max()
mn  = df[colunas_num].min()

tab_out = pd.DataFrame({
    "Coluna":        colunas_num,
    "Mínimo":        mn.values,
    "P01":           p01.values,
    "P99":           p99.values,
    "Máximo":        mx.values,
    "Máx / P99 (x)": (mx / p99.replace(0, np.nan)).values,
}).round(2).sort_values("Máx / P99 (x)", ascending=False)

print("
OUTLIERS EXTREMOS — razão entre Máximo e P99:")
display(tab_out.style.hide(axis="index"))



In [ ]:
# Detecção de problemas específicos
problemas = []

# Idade
idade_min, idade_max = df["idade"].min(), df["idade"].max()
if idade_min < 16:  problemas.append(f"Idade MÍNIMA = {idade_min} anos — impossível legalmente (erro de dado).")
if idade_max > 100: problemas.append(f"Idade MÁXIMA = {idade_max} anos — raro (top-capping recomendado no P99 = {p99['idade']:.0f}).")

# Uso limite rotativo
qtd_ultra_1 = (df["uso_limite_rotativo"] > 1).sum()
problemas.append(f"uso_limite_rotativo > 1 em {qtd_ultra_1:,} clientes ({qtd_ultra_1/len(df)*100:.2f}%) — ultrapassaram o limite contratado; informação real de risco NÃO dropar, mas max = {mx['uso_limite_rotativo']:.2f} pode ser winsorizada.")

# Renda
problemas.append(f"renda_mensal MÁXIMA = R$ {mx['renda_mensal']:,.0f} contra P99 = R$ {p99['renda_mensal']:,.0f} ({mx['renda_mensal']/p99['renda_mensal']:.1f}× o percentil) — outlier extremo; winsorizar no P99.")

# Dependentes
qtd_dep_alto = (df["dependentes"] > 10).sum()
if qtd_dep_alto > 0:
    problemas.append(f"dependentes > 10 = {qtd_dep_alto} casos contra P99 = {p99['dependentes']:.0f} — valor raro, tratar com top-capping.")

# Atrasos (3 faixas): valor MÁXIMO = 98 em todas! É impossível ter 98 atrasos de 30-59 dias em 2 anos.
for col_atraso in ["atrasos_30_59_dias", "atrasos_60_89_dias", "atrasos_90_mais_dias"]:
    p99_c  = p99[col_atraso]
    max_c  = mx[col_atraso]
    razao_c = max_c / (p99_c or np.nan)
    problemas.append(f"{col_atraso}: P99={p99_c:.0f}  vs  MÁXIMO={max_c:.0f} (≈ {razao_c:.1f}×) — 98 é fortemente suspeito de ser 'código de missing/erro de dados' (top-capping no P99 recomendado).")

# Financiamentos imobiliarios
qtd_fin_alto = (df["financiamentos_imobiliarios"] > 10).sum()
if qtd_fin_alto > 0:
    problemas.append(f"financiamentos_imobiliarios > 10 = {qtd_fin_alto} clientes; P99 = {p99['financiamentos_imobiliarios']:.0f} vs MÁXIMO = {mx['financiamentos_imobiliarios']:.0f} — valor extremo.")

# Duplicatas
dup = df.duplicated().sum()
problemas.append(f"Linhas DUPLICADAS = {dup:,} ({dup/len(df)*100:.2f}% da base) — investigar e remover duplicatas idênticas na Fase 3.")

from IPython.display import Markdown
md = "### Problemas específicos detectados (evidência por item)

"
for i, p in enumerate(problemas, 1):
    md += f"{i}. {p}

"
display(Markdown(md))



In [ ]:
# Visualizações (boxplots e histogramas)
cols_plot = ["idade", "uso_limite_rotativo", "razao_divida",
             "atrasos_30_59_dias", "atrasos_60_89_dias", "atrasos_90_mais_dias"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, c in enumerate(cols_plot):
    sns.boxplot(y=df[c].dropna(), ax=axes[i], color="#3498db", fliersize=2)
    axes[i].set_title(f"Boxplot — {c}", fontsize=11)
    axes[i].set_ylabel("")
plt.tight_layout()
fig.savefig(os.path.join(FIG_PATH, "fase02_boxplots_outliers.png"), dpi=120)
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, c in enumerate(cols_plot):
    sns.histplot(df[c].dropna(), kde=True, ax=axes[i], bins=40, color="#9b59b6")
    axes[i].set_title(f"Distribuição — {c}", fontsize=11)
    axes[i].set_xlabel("")
plt.tight_layout()
fig.savefig(os.path.join(FIG_PATH, "fase02_histogramas.png"), dpi=120)
plt.show()



## 9. Investigação complementar: Missing de renda + código 98 nas colunas de atraso
### 9.1 Inadimplência comparada: quem informou renda vs. quem NÃO informou

In [ ]:
# ============================================================
# ITEM 9.1 — Missing de renda x taxa de inadimplência
# ============================================================
df["renda_informada_flag"] = ~df["renda_mensal"].isnull()

tabela_renda = (
    df.groupby("renda_informada_flag")[alvo]
      .agg(
          Total="count",
          Inadimplentes="sum",
          Taxa_Inadimplencia=lambda s: (s.mean() * 100).round(2),
      )
      .reset_index()
)
tabela_renda["renda_informada_flag"] = tabela_renda["renda_informada_flag"].map(
    {True: "SIM — informou renda", False: "NÃO — missing de renda"}
)
tabela_renda.columns = ["Grupo", "N Clientes", "N Inadimplentes", "Taxa Inadimplência (%)"]
display(tabela_renda.style.hide(axis="index"))

taxa_sim = tabela_renda.loc[tabela_renda["Grupo"].str.startswith("SIM"), "Taxa Inadimplência (%)"].values[0]
taxa_nao = tabela_renda.loc[tabela_renda["Grupo"].str.startswith("NÃO"), "Taxa Inadimplência (%)"].values[0]
diff_pp = taxa_nao - taxa_sim

print(f"Taxa quem INFORMOU renda:    {taxa_sim:.2f}%")
print(f"Taxa quem NÃO informou renda: {taxa_nao:.2f}%")
print(f"DIFERENÇA (não-informou − informou) = {diff_pp:.2f} pp.")
print(f"Razão entre taxas (não/informou)   = {taxa_nao/taxa_sim:.2f}×")

# Gráfico
fig, ax = plt.subplots(figsize=(7, 4.8))
sns.barplot(x="Grupo", y="Taxa Inadimplência (%)", data=tabela_renda, ax=ax, palette=["#2ecc71", "#e74c3c"])
ax.set_title("Taxa de inadimplência: renda informada vs. missing", fontsize=13, pad=15)
for i, (v, n) in enumerate(zip(tabela_renda["Taxa Inadimplência (%)"].values, tabela_renda["N Clientes"].values)):
    ax.text(i, v + 0.08, f"{v:.2f}%\n(n={n:,})", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIG_PATH, "fase02_inadimplencia_renda_informada.png"), dpi=120)
plt.show()


### 9.2 Código suspeito 98 nas 3 colunas de atraso — investigação completa

In [ ]:
# ============================================================
# ITEM 9.2.1 — Distribuição de valores nas colunas de atraso (suspeitos >= 90)
# ============================================================
col_atrasos = [
    "atrasos_30_59_dias",
    "atrasos_60_89_dias",
    "atrasos_90_mais_dias",
]

for col in col_atrasos:
    vc = df[col].value_counts().sort_index()
    print(f"\n>>> {col} — valores únicos = {df[col].nunique()}")
    print("    Top 5 mais frequentes:")
    for v, c in vc.head(5).items():
        print(f"      valor={int(v):>3}  →  n={c:>7,}  ({c/len(df)*100:.3f}%)")
    suspeitos = vc[vc.index >= 90]
    if not suspeitos.empty:
        print("    Valores SUSPEITOS (>= 90 — impossível em 2 anos):")
        for v, c in suspeitos.items():
            print(f"      valor={int(v):>3}  →  n={c:>7,}  ({c/len(df)*100:.3f}%)")


In [ ]:
# ============================================================
# ITEM 9.2.2 — Sobreposição: as 264 linhas com 98 são AS MESMAS nas 3 colunas?
# ============================================================
mask_98_30 = df["atrasos_30_59_dias"] == 98
mask_98_60 = df["atrasos_60_89_dias"] == 98
mask_98_90 = df["atrasos_90_mais_dias"] == 98
mask_98_qualquer = mask_98_30 | mask_98_60 | mask_98_90
mask_98_todas    = mask_98_30 & mask_98_60 & mask_98_90

tabela_98_qtd = pd.DataFrame([
    ["Linhas com 98 em atrasos_30_59_dias",  mask_98_30.sum(),  (mask_98_30.mean()*100).round(3)],
    ["Linhas com 98 em atrasos_60_89_dias",  mask_98_60.sum(),  (mask_98_60.mean()*100).round(3)],
    ["Linhas com 98 em atrasos_90_mais_dias", mask_98_90.sum(), (mask_98_90.mean()*100).round(3)],
    ["Linhas com 98 EM QUALQUER das 3",      mask_98_qualquer.sum(), (mask_98_qualquer.mean()*100).round(3)],
    ["Linhas com 98 NAS 3 AO MESMO TEMPO",   mask_98_todas.sum(),   (mask_98_todas.mean()*100).round(3)],
], columns=["Medida", "N linhas", "%"])
display(tabela_98_qtd.style.hide(axis="index"))

# Combinações exatas
df_padrao = pd.DataFrame({
    "tem_30":  mask_98_30.astype(int),
    "tem_60":  mask_98_60.astype(int),
    "tem_90":  mask_98_90.astype(int),
})
df_padrao["Padrão"] = df_padrao.apply(lambda r:
    (["30-59"] if r["tem_30"] else []) +
    (["60-89"] if r["tem_60"] else []) +
    (["90+"]   if r["tem_90"] else []), axis=1
).apply(lambda x: ", ".join(x) if x else "nenhuma (sadio)")

combos = (df_padrao.groupby("Padrão")
                   .size()
                   .reset_index(name="n_linhas")
                   .sort_values("n_linhas", ascending=False)
)
combos["%"] = (combos["n_linhas"] / len(df) * 100).round(3)
print("\n>>> Combinações de colunas que contêm 98 (são duas únicas):")
display(combos.style.hide(axis="index"))

print("\n>>> 10 exemplos de linhas com 98 (para inspecionar):")
col_exibir = [alvo, "idade", "renda_mensal", "uso_limite_rotativo", "razao_divida"] + col_atrasos
display(df.loc[mask_98_qualquer, col_exibir].head(10))


### 9.3 Taxa de inadimplência do grupo com código 98 vs. resto da base

In [ ]:
# ============================================================
# ITEM 9.3 — Inadimplência do grupo com 98 vs. RESTO
# ============================================================
grupo = pd.DataFrame({
    "grupo": np.where(mask_98_qualquer, "COM 98 em pelo menos 1 coluna de atraso", "RESTO (sem 98)"),
    alvo: df[alvo].values,
})

tab = (grupo.groupby("grupo")[alvo]
            .agg(N="count", Inadimplentes="sum",
                 Taxa=lambda s: (s.mean() * 100).round(2))
            .reset_index()
      )
tab.columns = ["Grupo", "N Clientes", "N Inadimplentes", "Taxa Inadimplência (%)"]
display(tab.style.hide(axis="index"))

taxa_resto = tab.loc[tab["Grupo"].str.contains("RESTO"), "Taxa Inadimplência (%)"].values[0]
taxa_c98  = tab.loc[tab["Grupo"].str.contains("COM 98"),  "Taxa Inadimplência (%)"].values[0]

print(f"RESTO (sem 98)          : {taxa_resto:.2f}%")
print(f"GRUPO COM 98 em atrasos : {taxa_c98:.2f}%")
print(f"DIFERENÇA (grupo 98 − resto) = {taxa_c98 - taxa_resto:.2f} pp")
print(f"Razão entre taxas         = {taxa_c98/taxa_resto:.2f}×")

# Detalhe por padrão
tab2 = df_padrao[[alvo] if False else ["Padrão"]].copy()
tab2[alvo] = df[alvo].values
tab2 = (tab2.groupby("Padrão")[alvo]
             .agg(N="count", Inadimplentes="sum",
                  Taxa=lambda s: (s.mean() * 100).round(2))
             .reset_index()
       )
tab2.columns = ["Padrão (colunas com 98)", "N", "Inadimplentes", "Taxa Inadimplência (%)"]
print("\n>>> Detalhe por padrão de ocorrência do 98:")
display(tab2.style.hide(axis="index"))

# Gráfico
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(x="Padrão (colunas com 98)", y="Taxa Inadimplência (%)", data=tab2, ax=ax, palette="coolwarm")
ax.set_title("Taxa de inadimplência por padrão de 98 nas colunas de atraso", fontsize=13, pad=15)
for i, (v, n) in enumerate(zip(tab2["Taxa Inadimplência (%)"].values, tab2["N"].values)):
    ax.text(i, v + 0.08, f"{v:.2f}%\n(n={n:,})", ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=15)
plt.tight_layout()
fig.savefig(os.path.join(FIG_PATH, "fase02_inadimplencia_codigo_98_atrasos.png"), dpi=120)
plt.show()


### 9.4 Conclusões práticas (só com base nos números)

**Missing de renda:**
- Ao contrário da intuição inicial, quem **não informa** renda tem *menor* risco: **5,61%** contra **6,95%** de quem informa.
- Diferença real de **−1,34 pp** (razão 0,81×). Logo, a informação de missing ainda é útil como flag, mas deve ser interpretada como 'perfil distinto' — não como 'perfil mais arriscado'.
- Dropar essas 29.731 linhas continua **NÃO recomendado**: além de perder volume, perde-se um perfil que tem menor taxa de inadimplência e ajudaria o modelo a discriminar.

**Código 98:**
- **Padrão perfeito de código artificial:** as mesmas **264 linhas** apresentam 98 nas 3 colunas de atraso ao mesmo tempo. Também existem 5 linhas com 96 seguindo o mesmo padrão.
- Esse grupo tem **54,17% de inadimplência** contra 6,60% do resto — diferença de **+47,57 pp**, razão **8,21×** mais risco.
- **Ação recomendada para a Fase 3:** NÃO winsorize cegamente 98→P99 nas colunas de atraso. Isso apagaria o sinal preditivo mais forte de toda a base.
  Faça antes: (1) criar **flag binária `cod_erro_atrasos_98`** (1 quando as 3 colunas têm 98) e (2) substituir o 98 por NaN ou por P99 nas colunas originais de atraso, deixando a flag carregar o sinal desse perfil altíssimo de risco.

## 10. Confirmação visual: faixas de atrasos históricos × taxa de inadimplência

Para cada uma das 3 colunas de atraso (`30-59`, `60-89`, `≥90` dias) nós:
- Agrupamos em faixas: `0, 1, 2, 3, 4, 5+` e uma categoria **`Cód. sistema (96/98)`** isolada.
- Plotamos volume por faixa (barras, eixo esq.) e taxa de inadimplência (linha, eixo dir.), com linha pontilhada na taxa média geral (6,68%).
- Medimos a correlação de Spearman (ignorando 96/98 para não inflar artificialmente).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

ALVO = "inadimplente_2anos"

def bucketizar(v, max_val_real=4):
    if pd.isna(v):
        return "NA"
    if v in (96, 98):
        return "Cód. sistema (96/98)"
    if v <= max_val_real:
        return str(int(v))
    return f"{max_val_real+1}+"

col_atrasos = {
    "atrasos_30_59_dias":   "Atrasos 30–59 dias",
    "atrasos_60_89_dias":   "Atrasos 60–89 dias",
    "atrasos_90_mais_dias": "Atrasos ≥ 90 dias",
}
ORDEM = ["0", "1", "2", "3", "4", "5+", "Cód. sistema (96/98)"]
PALETA = ["#2ecc71", "#b2f0a0", "#fff066", "#ffc145", "#ff8c42", "#ff5e5e", "#8e44ad"]


In [ ]:
taxa_media = df[ALVO].mean() * 100
resultados = {}

for col, titulo in col_atrasos.items():
    bucket_col = f"{col}_faixa"
    df[bucket_col] = df[col].apply(bucketizar)
    tab = (df.groupby(bucket_col, dropna=False)[ALVO]
             .agg(n_clientes="count",
                  n_inadimplentes="sum",
                  taxa=lambda s: s.mean() * 100)
             .reset_index())
    tab.columns = ["Faixa", "N clientes", "N inadimplentes", "Taxa (%)"]
    tab["ordem"] = tab["Faixa"].apply(lambda x: ORDEM.index(x) if x in ORDEM else 999)
    tab = tab.sort_values("ordem").drop(columns="ordem").reset_index(drop=True)
    tab_disp = tab.round({"Taxa (%)": 2})
    resultados[col] = tab_disp
    print(f"\n=== {titulo} ({col}) ===")
    print(tab_disp.to_string(index=False))

    xs = list(range(len(tab)))
    labels = tab["Faixa"].tolist()
    cores = [PALETA[ORDEM.index(l)] if l in ORDEM else "#999" for l in labels]

    fig, ax1 = plt.subplots(figsize=(8.5, 5.))
    ax2 = ax1.twinx()
    bars = ax1.bar(xs, tab["N clientes"], color=cores, alpha=0.9)
    for b in bars:
        ax1.text(b.get_x() + b.get_width()/2, b.get_height(), f"{b.get_height():,.0f}",
                 ha="center", va="bottom", fontsize=8)
    linha, = ax2.plot(xs, tab["Taxa (%)"], color="#c0392b", marker="o", markersize=8, linewidth=2.2)
    for i, t in enumerate(tab["Taxa (%)"].values):
        ax2.annotate(f"{t:.1f}%", (xs[i], t), xytext=(0, 10),
                     textcoords="offset points", ha="center", color="#c0392b", fontweight="bold")
    ax2.axhline(taxa_media, color="#2c3e50", ls=":", lw=1.3,
                label=f"Média geral = {taxa_media:.2f}%")
    ax1.set_xticks(xs); ax1.set_xticklabels(labels)
    ax1.set_ylabel("Nº de clientes (barras)")
    ax2.set_ylabel("Taxa de inadimplência % (linha)", color="#c0392b")
    ax2.set_ylim(0, max(65, tab["Taxa (%)"].max() * 1.2))
    ax1.set_title(f"{titulo}: volume × taxa de inadimplência por faixa", fontsize=12, pad=12)
    fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.95), fontsize=9)
    plt.tight_layout()
    fpath = os.path.join(FIG_PATH, f"fase02_faixas_{col}.png")
    fig.savefig(fpath, dpi=150, bbox_inches="tight")
    print(f"  -> gráfico salvo em: {fpath}")
    plt.show()


In [ ]:
# --- Gráfico conjunto para comparar as 3 colunas lado a lado ---
fig, axes = plt.subplots(3, 1, figsize=(9, 12))
for idx, (col, titulo) in enumerate(col_atrasos.items()):
    ax = axes[idx]
    tab = resultados[col]
    xs = list(range(len(tab)))
    labels = tab["Faixa"].tolist()
    cores = [PALETA[ORDEM.index(l)] if l in ORDEM else "#999" for l in labels]
    ax.bar(xs, tab["N clientes"], color=cores, alpha=0.9)
    ax2 = ax.twinx()
    ax2.plot(xs, tab["Taxa (%)"], color="#c0392b", marker="o", ms=7, lw=2.)
    for i, t in enumerate(tab["Taxa (%)"].values):
        ax2.annotate(f"{t:.1f}%", (xs[i], t), xytext=(0, 8),
                     textcoords="offset points", ha="center", color="#c0392b", fontweight="bold", fontsize=8.5)
    ax.set_xticks(xs); ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Nº clientes", fontsize=9)
    ax2.set_ylabel("Inadimplência %", color="#c0392b", fontsize=9)
    ax2.tick_params(axis="y", labelcolor="#c0392b")
    ax2.axhline(taxa_media, color="#2c3e50", ls=":", lw=1.1, alpha=0.7)
    ax.set_title(f"{titulo} ({col})", fontsize=11)
fig.suptitle("Histórico de atrasos × taxa de inadimplência (por faixa)", fontsize=14, y=1.005)
plt.tight_layout()
fpath = os.path.join(FIG_PATH, "fase02_faixas_atrasos_conjunto.png")
fig.savefig(fpath, dpi=150, bbox_inches="tight")
print(f"Gráfico conjunto salvo em: {fpath}")
plt.show()


In [ ]:
# --- Correlação Spearman entre cada atraso e inadimplência (ignora 96/98) ---
mask = (
    (df["atrasos_30_59_dias"] < 90) &
    (df["atrasos_60_89_dias"] < 90) &
    (df["atrasos_90_mais_dias"] < 90)
)
corrs = {}
for col, titulo in col_atrasos.items():
    rho = df.loc[mask, [col, ALVO]].corr(method="spearman").iloc[0, 1]
    corrs[titulo] = rho

tab_corr = pd.DataFrame(corrs.items(), columns=["Coluna de atraso", "Spearman com inadimplência"])
tab_corr = tab_corr.sort_values("Spearman com inadimplência", ascending=False).reset_index(drop=True)
tab_corr["Spearman com inadimplência"] = tab_corr["Spearman com inadimplência"].round(4)
display(tab_corr.style.hide(axis="index"))

print("\nInterpretação:")
for i, linha in tab_corr.iterrows():
    r = linha["Spearman com inadimplência"]
    force = (
        "desprezível"  if abs(r) < 0.10 else
        "fraca"         if abs(r) < 0.20 else
        "moderada"      if abs(r) < 0.40 else
        "forte"
    )
    print(f"  · {linha['Coluna de atraso']:<20s} — ρ = {r:+.4f} — associação {force}.")


### 10.1 Conclusão visual

As 3 colunas de atraso histórico carregam **sinal preditivo monotônico forte**:
- Taxa cresce de forma consistente com o número de atrasos (0 → 1 → 2 → 3 → 4 → 5+), tanto visualmente quanto nos números.
- **Ordem de força:** `≥90 dias` (ρ = +0,335) > `60–89 dias` (ρ = +0,268) > `30–59 dias` (ρ = +0,251). Quanto mais severo o atraso, melhor a previsão.
- `Cód. sistema (96/98)` aparece isoladamente como **segunda categoria mais perigosa** (≈54,6%) e deve ser mantido via flag binária separada.
- O atraso de **≥90 dias, 4 vezes ou mais**, chega a **67% de inadimplência** na base — quase chance certa de calote.


---  
**Fim da Fase 2.** Próxima: Fase 3 — Preparação dos Dados.